# Lab 03 · Architecture tools and our own MCP server

**Prepared by Mustafa Assaf** · 5–10 minute example

## Setup
Run this cell once. It loads the reusable helpers; the example follows below. This protocol example needs no model key.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'mcp']):
    %pip install -q "agentic-ai-coursekit[mcp] @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"

## Read project facts, then save a decision

A tool performs an operation. Our own MCP server offers two: read a tiny JSON project fixture and save a decision in a notebook list. Nothing is written to an external service.

The **official MCP SDK** supplies discovery, argument schemas, and an in-process connection. No port or model key is needed. The fictional course case has three developers and no independent deployments.

In [ ]:
import json
from mcp import Client as MCPClient
from mcp.server import MCPServer
server = MCPServer("Architecture tools")
saved = []
@server.tool()
def read_metrics() -> dict:
    return json.loads('{"team_size": 3, "independent_deploys": false}')
@server.tool()
def save_decision(choice: str) -> str:
    saved.append(choice)
    return choice

## Discover and call the tools

The client reads the JSON fields and saves the decision from our course rule. These calls are a fixed workflow; the next lab lets the model choose its next action.

In [ ]:
async with MCPClient(server) as connection:
    print([tool.name for tool in (await connection.list_tools()).tools])
    metrics = await connection.call_tool("read_metrics", {})
    print(metrics.content[0].text)
    result = await connection.call_tool("save_decision", {"choice": "modular monolith"})
    print(result.content[0].text)
assert saved == ["modular monolith"]

**Expected output:** two discovered tools, the project's JSON fields, then `modular monolith`. The check confirms the save action changed the list.

**Small exercise:** add a `service_name` field to the JSON fixture and inspect the returned data.

Reading JSON is structured data extraction, not interpretation of a design document. MCP exposes operations; it does not select the architecture or validate the saved choice. The course rule is a teaching policy, not general architecture advice.

[Official SDK client guide](https://py.sdk.modelcontextprotocol.io/client/) · [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture)